Ce notebook permet de lancer un LDA uniquement sur les titres du sommaire et nn plus tout le contenu. 
Le but est d'obtenir des résultats mons bruités avec un sommaire qui porte déjà l'info nécessaire. 

In [1]:
!uv pip install -q nltk gensim pyLDAvis unidecode matplotlib seaborn pandas pyarrow

In [2]:
from gensim.models import CoherenceModel, LdaModel, LdaMulticore
from gensim.utils import simple_preprocess
from pathlib import Path
import gensim
import gensim.corpora as corpora
import json
import matplotlib.pyplot  as plt
import numpy as np
import os
import pandas as pd
import pyLDAvis
import pyLDAvis.gensim
import pyLDAvis.gensim_models as gensimvis
import warnings

In [3]:
INTERMEDIATE_DATA_DIR="intermediate_data"

# Utils LDA

In [4]:
def lda_model(processed_texts, num_topics=5, passes=10):
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    warnings.filterwarnings('ignore')
    dictionary = corpora.Dictionary(processed_texts)
    corpus = [dictionary.doc2bow(text) for text in processed_texts]
    model = LdaMulticore(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42, passes=passes ,workers=10, eta='auto' ,chunksize=1000)
    for topic in model.print_topics(num_words=5):
        print(topic)
    return model, corpus, dictionary
    
def visualize_lda(model, corpus, dictionary):
    pyLDAvis.enable_notebook()
    vis_data = gensimvis.prepare(model, corpus, dictionary)
    return pyLDAvis.display(vis_data)

In [5]:
def compute_coherence_values(dictionary, corpus, texts, max_topics=10):
    coherence_scores = []
    for num_topics in range(2, max_topics + 1):
        lda_model = LdaMulticore(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42, passes=5 ,workers=10, eta='auto' ,chunksize=1000)
        coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
        coherence_score = coherence_model.get_coherence()
        coherence_scores.append((num_topics, coherence_score))
        print(f"Num Topics: {num_topics}, Coherence Score: {coherence_score:.4f}")
    
    return coherence_scores

# Pour HS

In [6]:
df_hs = pd.read_parquet(f"{INTERMEDIATE_DATA_DIR}/processed_titles_hs.parquet")


In [7]:
all_chunks_hs = [list(title) for summary in df_hs["lda_documents"] for title in summary]


In [8]:
model_hs, corpus_hs, dictionary_hs = lda_model(all_chunks_hs)

(0, '0.134*"temp" + 0.129*"travail" + 0.050*"jours" + 0.037*"organisation" + 0.020*"partiel"')
(1, '0.041*"congés" + 0.041*"application" + 0.034*"champ" + 0.030*"droit" + 0.030*"rémunération"')
(2, '0.085*"travail" + 0.082*"durée" + 0.081*"accord" + 0.030*"entreprise" + 0.022*"hebdomadaire"')
(3, '0.043*"repos" + 0.043*"période" + 0.039*"jours" + 0.038*"absence" + 0.032*"cours"')
(4, '0.162*"heures" + 0.101*"supplémentaires" + 0.044*"contingent" + 0.029*"salariés" + 0.027*"annuel"')


In [9]:
visualize_lda(model_hs, corpus_hs, dictionary_hs)

In [10]:
compute_coherence_values(dictionary_hs, corpus_hs, all_chunks_hs, max_topics=20)

Num Topics: 2, Coherence Score: 0.3288
Num Topics: 3, Coherence Score: 0.3545
Num Topics: 4, Coherence Score: 0.3877
Num Topics: 5, Coherence Score: 0.3289
Num Topics: 6, Coherence Score: 0.3678
Num Topics: 7, Coherence Score: 0.3405
Num Topics: 8, Coherence Score: 0.3638
Num Topics: 9, Coherence Score: 0.3801
Num Topics: 10, Coherence Score: 0.3809
Num Topics: 11, Coherence Score: 0.3823
Num Topics: 12, Coherence Score: 0.3863
Num Topics: 13, Coherence Score: 0.3962
Num Topics: 14, Coherence Score: 0.3901
Num Topics: 15, Coherence Score: 0.4036
Num Topics: 16, Coherence Score: 0.3998
Num Topics: 17, Coherence Score: 0.4108
Num Topics: 18, Coherence Score: 0.3991
Num Topics: 19, Coherence Score: 0.4174
Num Topics: 20, Coherence Score: 0.4116


[(2, 0.32881237135992014),
 (3, 0.35451537698541197),
 (4, 0.3876698148912711),
 (5, 0.3288606574390373),
 (6, 0.3678201841165211),
 (7, 0.34045755551310236),
 (8, 0.36376142971860437),
 (9, 0.3800771107095664),
 (10, 0.3808740590531333),
 (11, 0.3822804069956574),
 (12, 0.3862838333147775),
 (13, 0.3962221618464709),
 (14, 0.39010156054306994),
 (15, 0.40359200135174106),
 (16, 0.39984113880733085),
 (17, 0.41084268740906404),
 (18, 0.3990884575943265),
 (19, 0.41735297998473153),
 (20, 0.4115997551542299)]

In [11]:
model_hs, corpus_hs, dictionary_hs = lda_model(all_chunks_hs, num_topics= 13)

(0, '0.112*"jours" + 0.084*"travail" + 0.080*"temp" + 0.071*"forfait" + 0.031*"amenagement"')
(1, '0.081*"congés" + 0.051*"prime" + 0.048*"payés" + 0.033*"journée" + 0.029*"modalités"')
(2, '0.158*"durée" + 0.143*"travail" + 0.060*"hebdomadaire" + 0.045*"repos" + 0.024*"maximale"')
(3, '0.124*"repos" + 0.074*"jours" + 0.058*"prise" + 0.045*"publicité" + 0.038*"dépôt"')
(4, '0.144*"rémunération" + 0.045*"période" + 0.039*"lissage" + 0.031*"départs" + 0.030*"annuelle"')
(5, '0.136*"temp" + 0.127*"travail" + 0.081*"application" + 0.078*"organisation" + 0.064*"champ"')
(6, '0.125*"période" + 0.092*"référence" + 0.084*"cours" + 0.083*"absence" + 0.036*"année"')
(7, '0.271*"heures" + 0.170*"supplémentaires" + 0.081*"contingent" + 0.051*"annuel" + 0.039*"supplementaires"')
(8, '0.137*"disposition" + 0.047*"cadre" + 0.028*"applicables" + 0.027*"finale" + 0.024*"relative"')
(9, '0.145*"temp" + 0.063*"travail" + 0.061*"salariés" + 0.061*"partiel" + 0.028*"pause"')
(10, '0.210*"accord" + 0.061*"e

In [12]:
visualize_lda(model_hs, corpus_hs, dictionary_hs)

Exception ignored in: <function ResourceTracker.__del__ at 0x7fddad8676a0>
Traceback (most recent call last):
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7fa24f2676a0>
Traceback (most recent call last):
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7fb5a653b6a0>
Traceback (most recent call last):
  File